# Volatility Forecasting with the Inait Forecasting API

This notebook demonstrates how to use the **Inait Forecasting API** to forecast stock return volatility, using AAPL and MSFT as concrete examples.

## What this notebook covers

1. **Single-target forecasting** — Predict future AAPL volatility up to a configurable horizon, using intraday and earnings-calendar features as exogenous inputs.
2. **Backtesting** — Evaluate model accuracy on a held-out historical period, with optional prediction intervals (e.g. 50% and 90% confidence bands).
3. **Benchmarking** — Compare the Inait model against a suite of baseline forecasters (e.g. Naive, Ridge, etc.) on the same backtest window.
4. **Cross Learning (global modeling)** — Train a single model jointly on AAPL and MSFT volatilities so that each series can inform the other, which is especially beneficial when the series share common dynamics.
5. **Explainability** — Inspect feature importance to understand which inputs and lags drive the model's predictions.

## Dataset

The input file (`dataset_GKYZ_2016_AAPL_MSFT_trimmed.csv`) contains daily GKYZ-style realised volatility estimates for AAPL and MSFT, together with a set of supplementary features:

| Feature group | Examples |
|---|---|
| Price-derived | `hl` (high-low range), `co` (close-open), `ocp` (overnight close-to-open), `log_returns`, `lret_squared` |
| Earnings calendar | `earning_horizon1` … `earning_horizon4` (days to next earnings announcement) |

## API workflow

Each operation follows the same three-step async pattern:

```
POST /prediction  (or /backtest, /benchmark)
  → session_id
GET  /status/{session_id}   (poll until "completed")
GET  /result/{session_id}   (fetch results)
```

When `run_explain=True` the result contains a `resource_id`; call `GET /download/{resource_id}` to retrieve the full output including SHAP explanations.

## Prerequisites

- A valid `auth_key` for the Inait Forecasting API (set in the cell below the imports).
- Python packages: `requests`, `pandas`, `plotly`, `dotenv`.

In [ ]:
from typing import Optional
import json
import requests
import time
import pandas as pd
from IPython.display import display
from dotenv import dotenv_values

import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# Plot pd.DataFrames with plotly
pd.options.plotting.backend = "plotly"

In [ ]:
# Load auth key from local credentials.txt file
config = dotenv_values("../credentials.txt")
base_url = config["API_BASE_URL"]
auth_key = config["API_AUTH_KEY"]

In [ ]:
input_data = "../data/dataset_GKYZ_2016_AAPL_MSFT_trimmed.csv"
# NOTE: check and change path accordingly

## Helper functions for making requests to the time series forecasting server and retrieving the results

In [ ]:
DEFAULT_REQUEST_TIMEOUT = 30


def _build_auth_headers(auth_key: Optional[str]) -> dict:
    """
    If an auth_key is provided, include BOTH common auth headers so the server
    can accept whichever it supports (Bearer or Azure APIM subscription key).
    """
    if not auth_key:
        return {}
    return {"Ocp-Apim-Subscription-Key": auth_key}


def make_request(url: str, payload: dict, auth_key: Optional[str] = None):
    """
    Makes a POST request to the specified endpoint with optional authentication.

    Args:
        url (str): The URL of the endpoint.
        payload (dict): The JSON payload for the request.
        auth_key (Optional[str]): Authentication key for the server.

    Returns:
        dict: The response JSON if the request is successful.

    Raises:
        Exception: If the request fails or returns a non-200 status code.
    """
    headers = {"Content-Type": "application/json"}
    headers |= _build_auth_headers(auth_key)
    response = requests.post(
        url, json=payload, headers=headers, timeout=DEFAULT_REQUEST_TIMEOUT
    )
    try:
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        raise Exception(
            f"Request failed: {response.status_code}, {response.text}"
        ) from e

    return response.json()


def make_get_request(
    url: str, session_id: Optional[str], auth_key: Optional[str] = None
):
    """
    Check task status and return result if completed.

    Args:
        url (str): Base server URL.
        session_id (str): Task session ID.
        auth_key (Optional[str]): Authentication key.

    Returns:
        dict: Task status or result if completed.
    """
    headers = _build_auth_headers(auth_key)
    # Check status
    full_url = f"{url}{session_id}" if session_id is not None else url
    status_response = requests.get(
        full_url, headers=headers, timeout=DEFAULT_REQUEST_TIMEOUT
    )
    try:
        status_response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        raise Exception(
            f"Status check failed for url {full_url}: {status_response.status_code}, {status_response.text}"
        ) from e
    return status_response.json()

In [ ]:
def get_job_status(session_id: str) -> str:
    response_status_bkt = make_get_request(
        url=f"{base_url}/status/", session_id=session_id, auth_key=auth_key
    )
    status_bkt = response_status_bkt["status"]
    return status_bkt

In [ ]:
def download_resource(resource_id: str) -> dict:
    """Download a resource from the server by its resource ID. To be used with EXPLAIN=True"""
    headers = _build_auth_headers(auth_key)
    download_url = f"{base_url}/download/{resource_id}"
    response = requests.get(
        download_url,
        headers=headers,
        allow_redirects=True,
        timeout=DEFAULT_REQUEST_TIMEOUT,
    )
    res = json.loads(response.content)
    return res

In [ ]:
def fetch_results_with_polling(
    session_id: str, max_attempts: int = 60, polling_interval: int = 60
) -> Optional[dict]:
    """
    Polls the server for job status and fetches results when completed.

    Args:
        session_id (str): The session ID of the job to check.
        max_attempts (int): Maximum number of polling attempts.
        polling_interval (int): Time to wait between polls in seconds.

    By default, it will poll every minute for up to 60 minutes.

    Returns:
        Optional[dict]: The result data if the job is completed successfully, None otherwise.
    """
    for attempt in range(max_attempts):
        status_pred = get_job_status(session_id)
        print(
            f"Attempt {attempt + 1}/{max_attempts}: Status of prediction job: {status_pred}"
        )

        if status_pred == "completed":
            response_result = make_get_request(
                url=f"{base_url}/result/", session_id=session_id, auth_key=auth_key
            )
            display(response_result)
            return response_result
        elif status_pred == "failed":
            print("Job failed!")
            return None
        else:
            if attempt < max_attempts - 1:  # Don't sleep on the last attempt
                print(
                    f"Job still running... waiting {polling_interval} seconds before next check"
                )
                time.sleep(polling_interval)
    else:
        print(
            f"Job did not complete after {max_attempts} attempts ({max_attempts * polling_interval} seconds)"
        )

## Plotting functions

In [ ]:
def mae_barplot(mae: dict[str, float]) -> None:
    """
    This function creates a horizontal bar plot of Mean Absolute Error (MAE) for different models using Plotly. The bars are colored according to the model names, and the MAE values are displayed as text on the bars. The plot is sorted in ascending order of MAE, allowing for easy comparison of model performance.
    """
    df = pd.DataFrame(list(mae.items()), columns=["model", "MAE"]).sort_values(
        "MAE", ascending=True
    )
    # Stable color mapping per model (in sorted order)
    models = df["model"].tolist()
    palette = (
        px.colors.qualitative.Set2
        + px.colors.qualitative.Plotly
        + px.colors.qualitative.Dark24
        + px.colors.qualitative.Safe
    )
    color_map = {m: palette[i % len(palette)] for i, m in enumerate(models)}
    fig = px.bar(
        df,
        x="MAE",
        y="model",
        orientation="h",
        color="model",
        color_discrete_map=color_map,
        title="MAE by model",
        labels={"MAE": "MAE", "model": "Model"},
        text=df["MAE"].map(lambda v: f"{v:.4f}"),
    )
    fig.update_traces(
        textposition="outside",
        hovertemplate="Model=%{y}<br>MAE=%{x:.4f}<extra></extra>",
    )
    fig.update_layout(
        yaxis=dict(categoryorder="array", categoryarray=models),
        legend_title_text="Model",
    )
    fig.show()

In [ ]:
def plot_backtest(
    pred_df: pd.DataFrame,
    historical_data: pd.DataFrame,
    target: str,
    horizon: int = 1,
    level: int = 50,
) -> go.Figure:
    """
    This function creates a plot that combines the historical data and the forecasts for a specific target and forecasting horizon. The historical data is shown as a solid line, while the point forecasts are shown as a dashed line. The prediction interval is visualized as a shaded area between the lower and upper bounds. This allows for an easy comparison of the model's predictions against the actual historical values, as well as an understanding of the uncertainty associated with the forecasts.
    """
    pred_df_fh = pred_df[pred_df["forecasting_horizon"] == horizon]
    observation_length = pred_df_fh.shape[0]
    target_len = int(2 * pred_df_fh.shape[0])
    title = target
    historical_color = "rgba(70, 130, 180, 0.7)"
    predicted_color = "#228B22"
    band_fill = "rgba(60, 179, 113, 0.25)"
    width_line = 2.5  # if len(predicted_data) > 0 else 3.0
    col_point_forecast = f"{target}__target"

    lo_col = f"{target}_predicted-lo-{level}"
    hi_col = f"{target}_predicted-hi-{level}"
    fig = go.Figure()

    # Historical plot
    fig.add_trace(
        go.Scatter(
            x=historical_data.iloc[-target_len:].index,
            y=historical_data.iloc[-target_len:][target],
            mode="lines",
            name="Historical",
            showlegend=True,
            line=dict(color=historical_color, width=width_line),
        ),
    )

    # Prediction plot
    pred_x = pred_df_fh[-observation_length:].index
    pred_y = pred_df_fh[-observation_length:][col_point_forecast]
    fig.add_trace(
        go.Scatter(
            x=pred_x,
            y=pred_y,
            mode="lines",
            name="Point Forecast",
            showlegend=True,
            line=dict(color=predicted_color),
            legendgroup=f"pred_{title}",
        ),
    )
    # Prediction interval band
    lo_y = pred_df_fh[-observation_length:][lo_col]
    hi_y = pred_df_fh[-observation_length:][hi_col]

    # Add LO trace (invisible)
    fig.add_trace(
        go.Scatter(
            x=pred_x,
            y=lo_y,
            mode="lines",
            line=dict(width=0),
            hoverinfo="skip",
            showlegend=False,
            legendgroup=f"pred_{title}",
        ),
    )

    # Add HI trace with fill
    fig.add_trace(
        go.Scatter(
            x=pred_x,
            y=hi_y,
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor=band_fill,
            hoverinfo="skip",
            showlegend=True,
            legendgroup=f"pred_{title}",
            name=f"{level}% interval",
        ),
    )
    fig.update_xaxes(title_text="Date")
    fig.show()

    return fig

In [ ]:
def plot_cl_history_and_forecast(
    pred_df: pd.DataFrame, data_cl: pd.DataFrame
) -> go.Figure:
    """
    This function creates a plot that combines the historical data and the forecasts for both AAPL and MSFT volatilities. The historical data is shown as solid lines, while the forecasts are shown as dashed lines. A vertical dashed line separates the historical period from the forecast period, providing a clear visual distinction between the two.
    """
    # Wide predictions: columns `AAPL`, `MSFT`
    pred_wide = (
        pred_df.rename(columns={"Inait": "pred"})
        .pivot(index="ds", columns="unique_id", values="pred")
        .rename_axis(None, axis=1)
    )
    pred_wide.index = pd.to_datetime(pred_wide.index)

    # True data
    true_data = data_cl[["AAPL", "MSFT"]].copy()
    true_data.index = pd.to_datetime(true_data.index)

    # End of historical, start of predictions
    hist_end = true_data.index.max()
    pred_start = pred_wide.index.min()

    # Concatenate along time for each ticker
    aapl_series = pd.concat([true_data["AAPL"], pred_wide["AAPL"]]).sort_index()
    msft_series = pd.concat([true_data["MSFT"], pred_wide["MSFT"]]).sort_index()

    # Masks to split real vs predicted segments
    aapl_real = aapl_series[aapl_series.index <= hist_end]
    aapl_pred = aapl_series[aapl_series.index >= pred_start]
    msft_real = msft_series[msft_series.index <= hist_end]
    msft_pred = msft_series[msft_series.index >= pred_start]

    fig = go.Figure()

    aapl_color = "#1f77b4"  # blue
    msft_color = "#ff7f0e"  # orange

    # AAPL real (solid) and predicted (dashed)
    fig.add_trace(
        go.Scatter(
            x=aapl_real.index,
            y=aapl_real.values,
            name="AAPL (real)",
            mode="lines",
            line=dict(color=aapl_color, width=2),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=aapl_pred.index,
            y=aapl_pred.values,
            name="AAPL (pred)",
            mode="lines",
            line=dict(color=aapl_color, width=2, dash="dash"),
        )
    )

    # MSFT real (solid) and predicted (dashed)
    fig.add_trace(
        go.Scatter(
            x=msft_real.index,
            y=msft_real.values,
            name="MSFT (real)",
            mode="lines",
            line=dict(color=msft_color, width=2),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=msft_pred.index,
            y=msft_pred.values,
            name="MSFT (pred)",
            mode="lines",
            line=dict(color=msft_color, width=2, dash="dash"),
        )
    )

    # Vertical dashed separator at end of history (convert to native datetime)
    # Separator as a shape (robust against Timestamp ops)
    fig.add_shape(
        type="line",
        x0=hist_end,
        x1=hist_end,
        y0=0,
        y1=1,
        xref="x",
        yref="paper",
        line=dict(width=2, dash="dash", color="#666"),
    )
    fig.add_annotation(
        x=hist_end,
        y=1,
        yref="paper",
        text="history | forecast",
        showarrow=False,
        yshift=10,
    )

    fig.update_layout(
        title="AAPL and MSFT: history concatenated with forecast",
        xaxis_title="Date",
        yaxis_title="Volatility",
        legend_title_text="Series",
        hovermode="x unified",
    )
    fig.show()
    return fig

In [ ]:
def plot_feature_importance(df_plot1: pd.DataFrame) -> go.Figure:
    """
    This plot shows the overall importance of each feature in the model, averaged across all time steps and lags. It helps identify which features are most influential in driving the model's predictions, regardless of when they occur.
    """
    df_plot1 = (
        df_plot1.groupby("base_name", observed=True)[["base_name", "mean_abs_shap_sum"]]
        .mean(numeric_only=True)
        .sort_values(by="mean_abs_shap_sum", ascending=True)
        .reset_index()
    )
    # print(df_plot1)
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=df_plot1["mean_abs_shap_sum"],
            y=df_plot1["base_name"],
            orientation="h",
            hovertemplate="Impact: %{x:.3f}<extra></extra>",
            marker=dict(color="steelblue"),
        )
    )
    fig.update_layout(title_text="Importance based on features")
    fig.update_layout(
        yaxis=dict(categoryorder="array", categoryarray=df_plot1["base_name"]),
        template="plotly_dark",
    )
    fig.show()


def plot_feature_and_lag_importance(df_plot2: pd.DataFrame) -> go.Figure:
    """
    This plot shows the importance of features at specific lags, which can help identify not only which features are important, but also at which time lags they have the most impact on the forecast.
    """
    df_plot2 = (
        df_plot2.groupby("feature_name", observed=True)[
            ["feature_name", "abs_mean_shap"]
        ]
        .mean(numeric_only=True)
        .sort_values(by="abs_mean_shap", ascending=True)
        .head(10)
        .reset_index()
    )
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=df_plot2["abs_mean_shap"],
            y=df_plot2["feature_name"],
            orientation="h",
            hovertemplate="Impact: %{x:.3f}<extra></extra>",
            marker=dict(color="steelblue"),
        )
    )
    fig.update_layout(title_text="Importance based on features and lags")
    fig.show()

## Volatility forecasting example for AAPL stock

In [ ]:
# Read volatility dataset for AAPL
data = pd.read_csv(input_data, index_col=0)

In [ ]:
# Define features to be used for prediction / backtest
# These are additional features that are not forecasted, but should help forecasting the target column
appl_cols = [
    "AAPL__hl",
    "AAPL__co",
    "AAPL__ocp",
    "AAPL__log_returns",
    "AAPL__lret_squared",
    "AAPL__earning_horizon1",
    "AAPL__earning_horizon2",
    "AAPL__earning_horizon3",
    "AAPL__earning_horizon4",
]
features = ",".join(appl_cols)

In [ ]:
data[["AAPL"] + appl_cols]

#### Forecasting

In [ ]:
# Prediction request - Payload for forecasting
payload_forecast = {
    "data": data.to_dict(orient="split"),
    "config": {
        "operation": "forecast",
        "operation_arguments": {
            "operation_type": "forecast",
            "forecasting_horizon": 3,
            "targets": "AAPL",
            "features": features,
            "prediction_interval_levels": None,
            "prediction_stride": 1,
            "end_date": None,
            "run_explain": False,
            "include_genai_summary": False,
        },
    },
    "background": True,
}

In [ ]:
# Prediction request - Send request for job (API)
response_request = make_request(
    base_url + "/prediction", payload_forecast, auth_key=auth_key
)
session_id = response_request["response"]["session_id"]

In [ ]:
response_result = fetch_results_with_polling(session_id)

In [ ]:
# Prediction results dataframe
pd.DataFrame(**response_result["response"]["data"]["predictions"])

#### Backtesting

In [ ]:
# Backtest request
payload_backtest = {
    "data": data.to_dict(orient="split"),
    "config": {
        "operation": "backtest",
        "operation_arguments": {
            "operation_type": "backtest",
            "forecasting_horizon": 3,
            "targets": "AAPL",
            "features": features,
            "prediction_interval_levels": "50,90",
            "prediction_stride": 1,
            "run_explain": False,
            "include_genai_summary": False,
            "backtest_size": None,
            "start_date": "2016-11-01",
            "end_date": "2016-12-31",
        },
    },
    "background": True,
}

In [ ]:
response_request = make_request(
    base_url + "/backtest", payload_backtest, auth_key=auth_key
)
session_id_bkt = response_request["response"]["session_id"]

In [ ]:
response_result_bkt = fetch_results_with_polling(session_id_bkt)

In [ ]:
preds = response_result_bkt["response"]["data"]["predictions"]
scores = response_result_bkt["response"]["data"]["scores"]

In [ ]:
pred_df = pd.DataFrame(**preds[0])
pred_df_fh1 = pred_df[pred_df["forecasting_horizon"] == 1]

In [ ]:
fig = pred_df_fh1[
    ["AAPL__target", "AAPL_predicted-hi-50", "AAPL_predicted-lo-50"]
].plot()
fig.show()

#### Benchmarking against other models

Note: this will also include inait-best model as part of the benchmark

In [ ]:
# Benchmark request
payload_benchmark = {}
payload_benchmark["data"] = payload_backtest["data"].copy()
payload_benchmark["background"] = payload_backtest["background"]
payload_benchmark["config"] = {}
payload_benchmark["config"]["operation"] = "benchmark"
payload_benchmark["config"]["operation_arguments"] = {}
payload_benchmark["config"]["operation_arguments"]["operation_type"] = "benchmark"
payload_benchmark["config"]["operation_arguments"]["backtest_config"] = (
    payload_backtest["config"]["operation_arguments"].copy()
)
# payload_benchmark

In [ ]:
response_request = make_request(
    base_url + "/benchmark", payload_benchmark, auth_key=auth_key
)
session_id_bench = response_request["response"]["session_id"]

In [ ]:
response_result_bench = fetch_results_with_polling(session_id_bench)

In [ ]:
all_scores_bench = {}
MAE_score = {}
for model, scores in response_result_bench["response"]["data"]["model_scores"].items():
    all_scores = pd.DataFrame(**scores)
    all_scores_bench[model] = all_scores
    MAE_score[model] = all_scores.loc[
        all_scores["metric"] == "test_error", "value"
    ].iloc[0]

In [ ]:
scores_inait_bkt = pd.DataFrame(
    **response_result_bench["response"]["data"]["backtest"]["inait-best"]["scores"]
)
MAE_score["inait-best"] = scores_inait_bkt.loc[
    scores_inait_bkt["metric"] == "test_error", "value"
].iloc[0]

In [ ]:
MAE_score

In [ ]:
mae_barplot(MAE_score)

#### Visualize backtest results

In [ ]:
fig = plot_backtest(
    pred_df=pred_df,
    historical_data=data,
    target="AAPL",
    horizon=1,
    level=90,
)

In [ ]:
# You can inspect only the predictions for a specific forecasting horizon (e.g. 1 step ahead) using:
# pred_df[pred_df["forecasting_horizon"]==1]

### An example of Cross Learning: AAPL and MSFT volatilities

In this case, the model will learn from both AAPL and MSFT volatilities jointly, potentially improving the forecast accuracy for both series.
Cross Learning is particularly useful when the time series are related or share similar patterns, as is often the case with stocks in the same sector.
Cross Learning is sometimes called "global modeling".

If Cross Learning is used with features that are target specific, we need to provide this specification using the characters `__` (double "underscore") in the feature names. For example, a feature named `AAPL__hl` will be used only for forecasting AAPL volatility, while a feature named `hl` (without the prefix) would be used for forecasting both AAPL and MSFT volatilities.

In [ ]:
data_cl = pd.read_csv(input_data, index_col=0)

In [ ]:
data_cl

In [ ]:
features_CL = ",".join(data_cl.columns.difference(["AAPL", "MSFT"]))

In [ ]:
features_CL

In [ ]:
# Prediction request
payload_forecast_cl = {
    "data": data_cl.to_dict(orient="split"),
    "config": {
        "operation": "forecast",
        "operation_arguments": {
            "operation_type": "forecast",
            "forecasting_horizon": 10,
            "targets": "AAPL, MSFT",
            "features": features_CL,
            "prediction_interval_levels": None,
            "prediction_stride": 1,
            "end_date": None,
            "run_explain": True,  # <------ turn on explainability for CL forecast: this will require downloading the resource to access the explanations (and the other output data)
            "include_genai_summary": False,
        },
    },
    "background": True,
}

In [ ]:
response_request_cl = make_request(
    base_url + "/prediction", payload_forecast_cl, auth_key=auth_key
)
session_id_cl = response_request_cl["response"]["session_id"]

In [ ]:
response_result_cl = fetch_results_with_polling(session_id_cl)

In [ ]:
res_id_cl = response_result_cl["response"]["resource_id"]
result_data_cl = download_resource(res_id_cl)

In [ ]:
result_data_cl

In [ ]:
# Predicted data using CL
pred_df_cl = pd.DataFrame(**result_data_cl["predictions"])

In [ ]:
fig = plot_cl_history_and_forecast(pred_df_cl, data_cl.iloc[-60:])

#### Explainability of the forecasts

In [ ]:
res_id_cl = response_result_cl["response"]["resource_id"]
data = download_resource(res_id_cl)

In [ ]:
df_plot1 = pd.DataFrame(**data["explain"]["df_plot1_time_mean_collapsed"])
df_plot2 = pd.DataFrame(**data["explain"]["df_plot2_time_mean_expanded"])

In [ ]:
plot_feature_importance(df_plot1)

In [ ]:
plot_feature_and_lag_importance(df_plot2)